# Agricultural Breeding NER + RE Pipeline

**Architecture:** BERT/DeBERTa Pipeline (NER first -> RE second)

**Platform:** CUDA -> MPS -> CPU (auto fallback)

**Scoring:** Total = 0.4 * Score_NER + 0.6 * Score_RE

---
```
Raw Text -> [NER: Token Classification] -> Entities -> [RE: Entity Marker + Classification] -> Relations
```

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
# !pip install transformers datasets seqeval accelerate scikit-learn -q

In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import json, random, os
import numpy as np
from collections import Counter
from typing import List, Dict, Tuple

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score as sk_f1, precision_score, recall_score
from seqeval.metrics import (
    f1_score as seq_f1,
    precision_score as seq_precision,
    recall_score as seq_recall,
)

print(f"PyTorch {torch.__version__} | Transformers {__import__('transformers').__version__}")

In [ ]:
# ============================================================
# Cell 3: Device Selection (CUDA -> MPS -> CPU)
# ============================================================
def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        name = torch.cuda.get_device_name(0)
        mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
        print(f"CUDA: {name} ({mem:.1f} GB)")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
        print(f"Apple MPS")
    else:
        dev = torch.device("cpu")
        print(f"CPU only")
    return dev

DEVICE = get_device()

if DEVICE.type == "cuda":
    TRAIN_BS, EVAL_BS, USE_FP16 = 16, 32, True
elif DEVICE.type == "mps":
    TRAIN_BS, EVAL_BS, USE_FP16 = 8, 16, False
else:
    TRAIN_BS, EVAL_BS, USE_FP16 = 4, 8, False

print(f"Train BS: {TRAIN_BS} | Eval BS: {EVAL_BS} | FP16: {USE_FP16}")

In [ ]:
# ============================================================
# Cell 4: Configuration
# ============================================================
class Config:
    MODEL_NAME = "microsoft/deberta-v3-base"  # best English NER/RE performance
    # MODEL_NAME = "bert-base-uncased"          # lighter, more compatible
    # MODEL_NAME = "roberta-base"               # good balance

    MAX_LEN = 384
    NER_EPOCHS = 8
    RE_EPOCHS = 5
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    GRAD_ACCUM = 1 if DEVICE.type == "cuda" else 2
    SEED = 42

    DATA_PATH = "HW_train_data.json"
    NER_MODEL_DIR = "./ner_model"
    RE_MODEL_DIR = "./re_model"
    VAL_SIZE = 0.15

    ENTITY_TYPES = ["ABS","BIS","BM","CHR","CROP","CROSS",
                    "GENE","GST","MRK","QTL","TRT","VAR"]
    RELATION_TYPES = ["AFF","CON","HAS","LOI","OCI","USE"]

    NER_LABELS = ["O"] + [f"{p}-{t}" for t in ENTITY_TYPES for p in ["B","I"]]
    RE_LABELS = ["no_relation"] + RELATION_TYPES

cfg = Config()

random.seed(cfg.SEED); np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED)
if DEVICE.type == "cuda": torch.cuda.manual_seed_all(cfg.SEED)

print(f"Model: {cfg.MODEL_NAME}")
print(f"NER labels: {len(cfg.NER_LABELS)} | RE labels: {len(cfg.RE_LABELS)}")

In [ ]:
# ============================================================
# Cell 5: Load + Explore Data
# ============================================================
with open(cfg.DATA_PATH, "r") as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")
lens = [len(s["text"]) for s in data]
print(f"Text length: {min(lens)}-{max(lens)}, mean={np.mean(lens):.0f}, median={np.median(lens):.0f}")

e_counts = [len(s["entities"]) for s in data]
r_counts = [len(s["relations"]) for s in data]
print(f"Entities/sample: mean={np.mean(e_counts):.1f}, 0-entity samples: {sum(1 for c in e_counts if c==0)}")
print(f"Relations/sample: mean={np.mean(r_counts):.1f}, 0-relation samples: {sum(1 for c in r_counts if c==0)}")

ent_counter = Counter(e["label"] for s in data for e in s["entities"])
rel_counter = Counter(r["label"] for s in data for r in s["relations"])
print(f"\nEntities: {dict(ent_counter.most_common())}")
print(f"Relations: {dict(rel_counter.most_common())}")

In [ ]:
# ============================================================
# Cell 6: Train/Val Split
# ============================================================
train_data, val_data = train_test_split(data, test_size=cfg.VAL_SIZE, random_state=cfg.SEED)
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

---
## Part 1: Named Entity Recognition (NER)
BIO token classification

In [ ]:
# ============================================================
# Cell 7: NER Label Encoding
# ============================================================
ner_label2id = {l: i for i, l in enumerate(cfg.NER_LABELS)}
ner_id2label = {i: l for l, i in ner_label2id.items()}

def char_labels_to_bio(text: str, entities: List[Dict]) -> List[str]:
    labels = ["O"] * len(text)
    for ent in sorted(entities, key=lambda e: (e["start"], e["end"])):
        s, e, t = ent["start"], ent["end"], ent["label"]
        if s >= len(text) or e > len(text) or t not in cfg.ENTITY_TYPES:
            continue
        labels[s] = f"B-{t}"
        for i in range(s + 1, e):
            if i < len(text):
                labels[i] = f"I-{t}"
    return labels

print(f"NER labels: {len(ner_label2id)} types")

In [ ]:
# ============================================================
# Cell 8: NER Dataset
# ============================================================
ner_tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)

class NERDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        text = sample["text"]
        char_labels = char_labels_to_bio(text, sample["entities"])

        enc = self.tokenizer(text, truncation=True, padding="max_length",
                             max_length=self.max_len, return_offsets_mapping=True,
                             return_tensors="pt")

        token_labels = []
        for offset in enc["offset_mapping"][0]:
            cs, ce = offset.tolist()
            if cs == ce:
                token_labels.append(-100)
            else:
                lbl = char_labels[cs] if cs < len(char_labels) else "O"
                token_labels.append(ner_label2id.get(lbl, ner_label2id["O"]))

        return {
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "labels": torch.tensor(token_labels, dtype=torch.long),
        }

ner_train_ds = NERDataset(train_data, ner_tokenizer, cfg.MAX_LEN)
ner_val_ds = NERDataset(val_data, ner_tokenizer, cfg.MAX_LEN)
print(f"NER Train: {len(ner_train_ds)} | Val: {len(ner_val_ds)}")

In [ ]:
# ============================================================
# Cell 9: NER Model + Trainer
# ============================================================
ner_model = AutoModelForTokenClassification.from_pretrained(
    cfg.MODEL_NAME, num_labels=len(cfg.NER_LABELS),
    id2label=ner_id2label, label2id=ner_label2id,
).to(DEVICE)

data_collator = DataCollatorForTokenClassification(tokenizer=ner_tokenizer)

def compute_ner_metrics(p):
    preds, labels = np.argmax(p.predictions, axis=2), p.label_ids
    true_lbls, pred_lbls = [], []
    for ps, ls in zip(preds, labels):
        t, p_ = [], []
        for pi, li in zip(ps, ls):
            if li != -100:
                t.append(ner_id2label[li])
                p_.append(ner_id2label[pi])
        true_lbls.append(t); pred_lbls.append(p_)
    return {
        "precision": seq_precision(true_lbls, pred_lbls),
        "recall": seq_recall(true_lbls, pred_lbls),
        "f1": seq_f1(true_lbls, pred_lbls),
    }

ner_args = TrainingArguments(
    output_dir="./ner_ckpt", num_train_epochs=cfg.NER_EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=cfg.GRAD_ACCUM, learning_rate=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY, warmup_ratio=cfg.WARMUP_RATIO,
    fp16=USE_FP16 and DEVICE.type=="cuda", logging_steps=20,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="f1",
    greater_is_better=True, report_to="none", seed=cfg.SEED,
    dataloader_num_workers=0,
)

ner_trainer = Trainer(model=ner_model, args=ner_args, train_dataset=ner_train_ds,
                      eval_dataset=ner_val_ds, data_collator=data_collator,
                      compute_metrics=compute_ner_metrics)

print(f"NER params: {sum(p.numel() for p in ner_model.parameters())/1e6:.1f}M")
print(f"Effective BS: {TRAIN_BS * cfg.GRAD_ACCUM}")

In [ ]:
# ============================================================
# Cell 10: NER Training
# ============================================================
print("Training NER model...")
ner_trainer.train()
ner_trainer.save_model(cfg.NER_MODEL_DIR)
ner_tokenizer.save_pretrained(cfg.NER_MODEL_DIR)
print(f"NER model saved to {cfg.NER_MODEL_DIR}")

In [ ]:
# ============================================================
# Cell 11: NER Evaluation
# ============================================================
ner_res = ner_trainer.evaluate()
ner_p, ner_r, ner_f1 = ner_res["eval_precision"], ner_res["eval_recall"], ner_res["eval_f1"]
ner_score = 0.5 * ner_f1 + 0.25 * ner_p + 0.25 * ner_r
print(f"NER  Precision={ner_p:.4f}  Recall={ner_r:.4f}  F1={ner_f1:.4f}  Score={ner_score:.4f}")

---
## Part 2: Relation Extraction (RE)
Entity-marker + sequence classification

In [ ]:
# ============================================================
# Cell 12: RE Data Preparation
# ============================================================
def build_re_samples(samples: List[Dict], neg_ratio: float = 1.5) -> List[Dict]:
    re_samples = []
    for s in samples:
        text = s["text"]
        entities = s["entities"]
        relations = s["relations"]
        if len(entities) < 2: continue

        rel_map = {}
        for r in relations:
            rel_map[(r["head_start"],r["head_end"],r["tail_start"],r["tail_end"])] = r["label"]

        pos, neg = [], []
        for i, e1 in enumerate(entities):
            for j, e2 in enumerate(entities):
                if i == j: continue
                key = (e1["start"], e1["end"], e2["start"], e2["end"])
                item = {"text": text, "e1": e1, "e2": e2}
                if key in rel_map:
                    item["label"] = rel_map[key]
                    pos.append(item)
                else:
                    item["label"] = "no_relation"
                    neg.append(item)

        re_samples.extend(pos)
        n_neg = min(int(len(pos) * neg_ratio) + 2, len(neg))
        if n_neg > 0:
            re_samples.extend(random.sample(neg, n_neg))
    return re_samples

re_label2id = {l: i for i, l in enumerate(cfg.RE_LABELS)}
re_id2label = {i: l for l, i in re_label2id.items()}

re_train_raw = build_re_samples(train_data)
re_val_raw = build_re_samples(val_data)
print(f"RE samples -> Train: {len(re_train_raw)} | Val: {len(re_val_raw)}")
print(f"Distribution: {dict(Counter(r['label'] for r in re_train_raw))}")

In [ ]:
# ============================================================
# Cell 13: RE Dataset
# ============================================================
re_tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)
re_tokenizer.add_special_tokens({"additional_special_tokens": ["[E1]","[/E1]","[E2]","[/E2]"]})

def format_entity_markers(text: str, e1: Dict, e2: Dict) -> str:
    markers = [(e1["start"], " [E1] "), (e1["end"], " [/E1] "),
               (e2["start"], " [E2] "), (e2["end"], " [/E2] ")]
    markers.sort(key=lambda x: x[0])
    pieces, last = [], 0
    for pos, tok in markers:
        if pos > last: pieces.append(text[last:pos])
        pieces.append(tok)
        last = pos
    if last < len(text): pieces.append(text[last:])
    return "".join(pieces).strip()

class REDataset(Dataset):
    def __init__(self, re_samples, tokenizer, max_len):
        self.samples = re_samples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        marked = format_entity_markers(s["text"], s["e1"], s["e2"])
        enc = self.tokenizer(marked, truncation=True, padding="max_length",
                             max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "labels": torch.tensor(re_label2id[s["label"]], dtype=torch.long),
        }

re_train_ds = REDataset(re_train_raw, re_tokenizer, cfg.MAX_LEN)
re_val_ds = REDataset(re_val_raw, re_tokenizer, cfg.MAX_LEN)
print(f"RE Train: {len(re_train_ds)} | Val: {len(re_val_ds)}")

s0 = re_train_ds[0]
print(f"Sample label: {re_id2label[s0['labels'].item()]}")

In [ ]:
# ============================================================
# Cell 14: RE Model + Training
# ============================================================
re_model = AutoModelForSequenceClassification.from_pretrained(
    cfg.MODEL_NAME, num_labels=len(cfg.RE_LABELS),
    id2label=re_id2label, label2id=re_label2id,
).to(DEVICE)
re_model.resize_token_embeddings(len(re_tokenizer))

def compute_re_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    acc = (preds == labels).mean()
    mask = labels != re_label2id["no_relation"]
    f1 = sk_f1(labels[mask], preds[mask], average="micro", zero_division=0) if mask.sum() > 0 else 0.0
    return {"accuracy": acc, "f1": f1}

re_args = TrainingArguments(
    output_dir="./re_ckpt", num_train_epochs=cfg.RE_EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=cfg.GRAD_ACCUM, learning_rate=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY, warmup_ratio=cfg.WARMUP_RATIO,
    fp16=USE_FP16 and DEVICE.type=="cuda", logging_steps=20,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="f1",
    greater_is_better=True, report_to="none", seed=cfg.SEED,
    dataloader_num_workers=0,
)

re_trainer = Trainer(model=re_model, args=re_args, train_dataset=re_train_ds,
                     eval_dataset=re_val_ds, compute_metrics=compute_re_metrics)

print("Training RE model...")
re_trainer.train()
re_trainer.save_model(cfg.RE_MODEL_DIR)
re_tokenizer.save_pretrained(cfg.RE_MODEL_DIR)
print(f"RE model saved to {cfg.RE_MODEL_DIR}")

In [ ]:
# ============================================================
# Cell 15: RE Evaluation
# ============================================================
re_res = re_trainer.evaluate()
print(f"RE  Accuracy={re_res['eval_accuracy']:.4f}  F1(micro)={re_res['eval_f1']:.4f}")

rp = re_trainer.predict(re_val_ds)
from sklearn.metrics import classification_report
print("\nPer-class:")
print(classification_report(rp.label_ids, np.argmax(rp.predictions, axis=1),
                            target_names=cfg.RE_LABELS, zero_division=0))

---
## Part 3: Post-Processing Rules

In [ ]:
# ============================================================
# Cell 16: Post-Processing
# ============================================================
VALID_PAIRS = {
    "CON": [("CROP","VAR"),("CROP","TRT"),("CHR","QTL"),("CHR","MRK"),("CHR","GENE"),("QTL","MRK")],
    "USE": [("VAR","BM"),("CROP","BM"),("GENE","BM")],
    "HAS": [("VAR","TRT"),("CROP","TRT"),("GENE","TRT"),("QTL","TRT"),("CROSS","TRT")],
    "AFF": [("ABS","TRT"),("BIS","TRT"),("GENE","TRT"),("MRK","TRT"),("QTL","TRT")],
    "OCI": [("TRT","GST"),("ABS","GST"),("BIS","GST")],
    "LOI": [("MRK","CHR"),("QTL","CHR"),("GENE","CHR")],
}

def postprocess_relations(entities: List[Dict], relations: List[Dict]) -> List[Dict]:
    filtered, seen = [], set()
    for rel in relations:
        ht, tt, rl = rel.get("head_type",""), rel.get("tail_type",""), rel.get("label","")
        if (ht, tt) not in VALID_PAIRS.get(rl, []):
            continue
        key = (rel["head_start"],rel["head_end"],rel["tail_start"],rel["tail_end"],rl)
        if key in seen: continue
        seen.add(key)
        filtered.append(rel)
    return filtered

print("Post-processing rules loaded")

---
## Part 4: Full Pipeline Inference

In [ ]:
# ============================================================
# Cell 17: Pipeline Class
# ============================================================
class NERREPipeline:
    def __init__(self, ner_model, ner_tokenizer, re_model, re_tokenizer, device):
        self.ner_model = ner_model.eval()
        self.ner_tokenizer = ner_tokenizer
        self.re_model = re_model.eval()
        self.re_tokenizer = re_tokenizer
        self.device = device

    def predict_entities(self, text: str, threshold: float = 0.5) -> List[Dict]:
        enc = self.ner_tokenizer(text, truncation=True, padding=True,
                                 max_length=cfg.MAX_LEN, return_offsets_mapping=True,
                                 return_tensors="pt")
        ids = enc["input_ids"].to(self.device)
        mask = enc["attention_mask"].to(self.device)
        offsets = enc["offset_mapping"][0].tolist()

        with torch.no_grad():
            probs = torch.softmax(self.ner_model(ids, mask).logits[0], dim=-1)

        entities, cur = [], None
        for off, prob in zip(offsets, probs):
            cs, ce = off
            if cs == ce: continue
            pid = prob.argmax().item()
            conf = prob[pid].item()
            plabel = ner_id2label[pid]

            if plabel == "O":
                if cur:
                    cur["end"] = ce
                    cur["text"] = text[cur["start"]:cur["end"]]
                    if cur["confidence"] >= threshold: entities.append(cur)
                    cur = None
                continue

            prefix, etype = plabel.split("-")
            if prefix == "B":
                if cur:
                    cur["end"] = ce
                    cur["text"] = text[cur["start"]:cur["end"]]
                    if cur["confidence"] >= threshold: entities.append(cur)
                cur = {"start": cs, "label": etype, "confidence": conf}
            elif prefix == "I" and cur and cur["label"] == etype:
                cur["confidence"] = min(cur["confidence"], conf)

        if cur:
            cur["end"] = len(text)
            cur["text"] = text[cur["start"]:cur["end"]]
            if cur["confidence"] >= threshold: entities.append(cur)
        return entities

    def predict_relation(self, text: str, e1: Dict, e2: Dict) -> Tuple[str, float]:
        marked = format_entity_markers(text, e1, e2)
        enc = self.re_tokenizer(marked, truncation=True, padding=True,
                                max_length=cfg.MAX_LEN, return_tensors="pt")
        ids = enc["input_ids"].to(self.device)
        mask = enc["attention_mask"].to(self.device)

        with torch.no_grad():
            probs = torch.softmax(self.re_model(ids, mask).logits[0], dim=-1)

        pid = probs.argmax().item()
        return re_id2label[pid], probs[pid].item()

    def predict(self, text: str, ner_th: float = 0.45, re_th: float = 0.4) -> Dict:
        entities = self.predict_entities(text, threshold=ner_th)
        relations = []
        for i, e1 in enumerate(entities):
            for j, e2 in enumerate(entities):
                if i == j: continue
                rlabel, conf = self.predict_relation(text, e1, e2)
                if rlabel != "no_relation" and conf >= re_th:
                    relations.append({
                        "head": e1["text"], "head_start": e1["start"],
                        "head_end": e1["end"], "head_type": e1["label"],
                        "tail": e2["text"], "tail_start": e2["start"],
                        "tail_end": e2["end"], "tail_type": e2["label"],
                        "label": rlabel,
                    })
        relations = postprocess_relations(entities, relations)
        return {"text": text, "entities": entities, "relations": relations}

pipeline = NERREPipeline(ner_model, ner_tokenizer, re_model, re_tokenizer, DEVICE)
print("Pipeline ready!")

# Quick test
res = pipeline.predict("VrFRO8 reduces SOD contents under salt stress.")
print(f"Entities: {[(e['label'], e['text']) for e in res['entities']]}")
print(f"Relations: {[(r['label'], r['head'], r['tail']) for r in res['relations']]}")

In [ ]:
# ============================================================
# Cell 18: Full Evaluation with Custom Scoring
# ============================================================
def evaluate_pipeline(pipeline, val_samples, ner_th=0.45, re_th=0.4):
    ner_tp = ner_fp = ner_fn = re_tp = re_fp = re_fn = 0

    for s in val_samples:
        pred = pipeline.predict(s["text"], ner_th=ner_th, re_th=re_th)

        gold_ent = {(e["start"],e["end"],e["label"]) for e in s["entities"]}
        pred_ent = {(e["start"],e["end"],e["label"]) for e in pred["entities"]}
        ner_tp += len(gold_ent & pred_ent)
        ner_fp += len(pred_ent - gold_ent)
        ner_fn += len(gold_ent - pred_ent)

        gold_rel = {(r["head_start"],r["head_end"],r["tail_start"],r["tail_end"],r["label"])
                     for r in s["relations"]}
        pred_rel = {(r["head_start"],r["head_end"],r["tail_start"],r["tail_end"],r["label"])
                     for r in pred["relations"]}
        re_tp += len(gold_rel & pred_rel)
        re_fp += len(pred_rel - gold_rel)
        re_fn += len(gold_rel - pred_rel)

    def calc(p, r):
        f1 = 2*p*r/(p+r) if p+r>0 else 0
        return {"Precision": p, "Recall": r, "F1": f1, "Score": 0.5*f1+0.25*p+0.25*r}

    np_ = ner_tp/(ner_tp+ner_fp) if ner_tp+ner_fp>0 else 0
    nr_ = ner_tp/(ner_tp+ner_fn) if ner_tp+ner_fn>0 else 0
    rp_ = re_tp/(re_tp+re_fp) if re_tp+re_fp>0 else 0
    rr_ = re_tp/(re_tp+re_fn) if re_tp+re_fn>0 else 0

    ner_m = calc(np_, nr_)
    re_m = calc(rp_, rr_)
    total = 0.4 * ner_m["Score"] + 0.6 * re_m["Score"]
    return {"NER": ner_m, "RE": re_m, "Total_Score": total}

print("Evaluating on first 50 val samples...")
results = evaluate_pipeline(pipeline, val_data[:50])
print(f"\n{'='*50}")
for task, m in results.items():
    if task == "Total_Score":
        print(f"\n  TOTAL SCORE: {m:.4f}")
    else:
        print(f"\n  {task}: P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f} Score={m['Score']:.4f}")
print(f"{'='*50}")

---
## Part 5: Threshold Tuning (Optional)

In [ ]:
# ============================================================
# Cell 19: Threshold Tuning
# ============================================================
def tune_thresholds(pipeline, val_subset):
    best_score, best_params = -1, (0.45, 0.4)
    for ner_th in [0.35, 0.40, 0.45, 0.50, 0.55]:
        for re_th in [0.30, 0.35, 0.40, 0.45, 0.50]:
            r = evaluate_pipeline(pipeline, val_subset, ner_th, re_th)
            if r["Total_Score"] > best_score:
                best_score = r["Total_Score"]
                best_params = (ner_th, re_th)
    return best_params, best_score

print("Tuning on 30 samples...")
(ner_th, re_th), score = tune_thresholds(pipeline, val_data[:30])
print(f"Best: NER_th={ner_th} RE_th={re_th} Score={score:.4f}")

---
## Part 6: Test Set Prediction

In [ ]:
# ============================================================
# Cell 20: Predict on Test Data
# ============================================================
TEST_PATH = "HW_test_data.json"

def predict_test(pipeline, test_path, output_path, ner_th=0.45, re_th=0.4):
    if not os.path.exists(test_path):
        print(f"File not found: {test_path}")
        return
    with open(test_path) as f:
        test_data = json.load(f)
    results = []
    for i, s in enumerate(test_data):
        t = s["text"] if isinstance(s, dict) else s
        pred = pipeline.predict(t, ner_th=ner_th, re_th=re_th)
        results.append(pred)
        if (i+1) % 20 == 0:
            print(f"  {i+1}/{len(test_data)}")
    with open(output_path, "w") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved to {output_path}")

print("Ready. Set TEST_PATH then run predict_test(pipeline, TEST_PATH, 'predictions.json')")

In [ ]:
# ============================================================
# Cell 21: Model Reload (for later use)
# ============================================================
print(f"Models saved at:")
print(f"  NER -> {cfg.NER_MODEL_DIR}")
print(f"  RE  -> {cfg.RE_MODEL_DIR}")
print(f"\nTo reload later:")
print(f"  from transformers import AutoModelForTokenClassification, AutoModelForSequenceClassification, AutoTokenizer")
print(f"  ner_model = AutoModelForTokenClassification.from_pretrained('{cfg.NER_MODEL_DIR}').to(DEVICE)")
print(f"  ner_tokenizer = AutoTokenizer.from_pretrained('{cfg.NER_MODEL_DIR}')")
print(f"  re_model = AutoModelForSequenceClassification.from_pretrained('{cfg.RE_MODEL_DIR}').to(DEVICE)")
print(f"  re_tokenizer = AutoTokenizer.from_pretrained('{cfg.RE_MODEL_DIR}')")